# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## 05 · Vector Database

This notebook demonstrates the **Vector Database** stage of the RAG pipeline.

A vector database is used to store the embeddings generated from the document chunks, together with their associated metadata. It provides efficient similarity search over the embedding space, allowing the RAG system to retrieve the document chunks that are most relevant to a user's query. During retrieval, the query is also converted into an embedding, and the vector database identifies the stored embeddings that are closest to it according to a similarity measure.

For this project, **ChromaDB** was selected as the vector database. Chroma provides a simple Python interface for storing and querying embeddings and integrates naturally with the **LangChain** components used throughout the project. It also supports both **persistent storage** and **in-memory databases**, which makes it suitable for both local development and the deployment requirements of the *Streamlit* application.

The `VectorDatabase` class is defined in `src/github_rag/vectordatabase/chroma_store.py`.

#### **Notebook objectives**

* Create a local, persistent Chroma database stored on disk.
* Create an in-memory Chroma database.
* Store document embeddings and their associated metadata.
* Perform simple similarity retrieval tasks to understand how ChromaDB works in the context of a RAG system.

## Imports and Configuration

In [1]:
import time
from pathlib import Path
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.processor import Processor
from github_rag.chunking.chunker import Chunker
from github_rag.embedding.embedding_model import create_embedding_model
from github_rag.vectordatabase.chroma_store import ChromaStore

c:\Users\david\anaconda3\envs\github-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


You can explore any GitHub repository simply by providing its URL. For this demo, I'll be using one of my own end-to-end Data Science projects, which combines XGBoost and SHAP to build a predictive and explainable system for classifying urban typologies.

In [2]:
REPOSITORY_URL = "https://github.com/davidfernxndez/spark-active-learning"

## 1. Ingestion -> processing -> chunking

This section apply the Ingestion, processing and chunking phase to get the prepared *LangChain Documents* for Embedding and VectorDB storing.

In [3]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents ingested: {len(documents)}")

Number of documents ingested: 12


In [4]:
# Set the Processor Object
processor = Processor()

# Apply processing to the ingested documents
documents_processed = processor.process(documents)

print(f"Number of documents before processing: {len(documents_processed)}")

Number of documents before processing: 45


In [5]:
# Set the Chunker object that managed all the specific chunkers
chunker = Chunker()

# Apply chunking
documents_chunking = chunker.chunk(documents_processed)

print(f"Number of documents before chunking: {len(documents_chunking)}")

Number of documents before chunking: 183


## 2. Load the Embedding model

We first need to load the embedding model that will be passed to the ChromaDB instance. The same model used to generate the document embeddings must be used to embed user queries during the retrieval process.

In [6]:
# Set the HuggingFace embedding model name
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Load the model
embedding_model = create_embedding_model(embedding_model_name)

## 3. Create Chroma Database

The `ChromaStore` class supports two different storage modes depending on the execution environment. For local development, it can create a **persistent Chroma database** stored on disk. The database is stored in the `/data` directory at the root of the repository, allowing it to persist between executions and making it convenient for development and experimentation.

It also supports creating an **in-memory Chroma database**, which is intended for the Streamlit deployment. In this mode, the vector database exists only during the current application session and does not require persistent storage on disk.

### 3.1 Persistent Chroma database

In [7]:
start_time = time.time()
# Create ChromaDB instance for local disk storing
vectorDB_local = ChromaStore(
    mode="local",
    collection_name="github_repo"
)

# Create the database with documents and the embedding model
vectorDB_local.create(
    documents=documents_chunking,
    embedding_model=embedding_model
)
print(f"Database generation time: {time.time() - start_time:.2f} seconds")

# Check the number of stored documents
stored_documents = vectorDB_local.count()

print(f"Documents provided: {len(documents_chunking)}")
print(f"Documents stored:   {stored_documents}")

Database generation time: 4.17 seconds
Documents provided: 183
Documents stored:   183


We can verify that it has been created on disk.

In [8]:
persist_directory = Path("../data/chroma")

print("Database exists:", persist_directory.exists())
print("Files:")
for path in persist_directory.iterdir():
    print(path)

Database exists: True
Files:
..\data\chroma\10287bc0-71e7-4a14-8213-4c3d178c1fe7
..\data\chroma\chroma.sqlite3


### 3.1 In-Memory Chroma database

In [9]:
start_time = time.time()
# Create ChromaDB instance using in-memory storage
vectorDB_memory = ChromaStore(
    mode="memory",
    collection_name="github_repo_memory"
)

# Create the database
vectorDB_memory.create(
    documents=documents_chunking,
    embedding_model=embedding_model
)

print(f"Database generation time: {time.time() - start_time:.2f} seconds")

# Check the number of stored documents
stored_documents = vectorDB_memory.count()

print(f"Documents provided: {len(documents_chunking)}")
print(f"Documents stored:   {stored_documents}")

Database generation time: 3.95 seconds
Documents provided: 183
Documents stored:   183


The following demonstrates that the database doesn`t exist in a local directory.

In [10]:
# Verify that no persistence directory is configured
print(f"Persistence directory: {vectorDB_memory.persist_directory}")

Persistence directory: None


## 4. Perform Retrieval from ChromaDB

ChromaDB provides built-in methods for retrieving documents based on the similarity between a query and the stored embeddings. These methods allow us to perform semantic searches over the vector database and retrieve the document chunks that are most relevant to a given query.

In this section, we perform some simple retrieval examples to verify that the documents stored in ChromaDB can be successfully retrieved based on their semantic similarity to a user query.

The get method allows us to retrieve some documents and check their ID, metadata and text content.

In [11]:
results = vectorDB_local.get(limit=3)

for i in range(3):
    print(f"Document {i}")
    print("ID:", results["ids"][i])
    print("Metadata:", results["metadatas"][i])
    print("Content:", results["documents"][i][:500])
    print("-" * 80)
    print("\n")

Document 0
ID: 65406ee9-028a-43a7-a47a-ae2eff5a4593
Metadata: {'repository': 'davidfernxndez/spark-active-learning', 'chunk_index': 0, 'chunk_id': 'enviroment.yml::0', 'source': 'https://github.com/davidfernxndez/spark-active-learning/blob/main/enviroment.yml', 'file_type': '.yml', 'file_path': 'enviroment.yml', 'branch': 'main'}
Content: name: spark-active-learning
channels:
  - conda-forge
  - defaults
  - pytorch
  - nvidia

dependencies:
  - python=3.8.10
  - ipython=7.25.0
  - ipywidgets=7.6.3
  - numpy=1.21.1
  - matplotlib=3.4.2
  - nb_conda_kernels=2.3.1
  - pandas=1.3.0
  - scipy=1.7.0
  - scikit-learn=1.0.2
  - jupyter=1.0.0
  - jupyterlab=3.0.16
  - graphviz=2.48.0
  - pydotplus=2.0.2
  - dtreeviz=1.3.1
  - pyspark=3.3.0
  - pip=21.1.3
  - shapely=1.8.5
  - owslib=0.27.2
  - rasterio=1.3.3
  - geopandas=0.12.1
  - cont
--------------------------------------------------------------------------------


Document 1
ID: 7a1f8c59-2be4-4167-bf89-ec1f12b8bf10
Metadata: {'file_type':

We can do a simple RAG retrieval example using the `similarity_search_with_score` method.

In [12]:
# Verify semantic search
query = "¿Which clustering algorithm is used in diversity strategy?"

results = vectorDB_local.similarity_search_with_score(
    query=query,
    k=10
)

print(f"\nQuery: {query}\n")

for i, (document, distance) in enumerate(results, start=1):
    print(f"Result {i}")
    print(f"Distance: {distance:.4f}")
    print(f"Source: {document.metadata.get('source')}")
    print(f"File: {document.metadata.get('file_path')}")
    print(document.page_content)
    print("-" * 80)



Query: ¿Which clustering algorithm is used in diversity strategy?

Result 1
Distance: 8.7034
Source: https://github.com/davidfernxndez/spark-active-learning/blob/main/main.ipynb
File: main.ipynb
In this context, the goal of clustering is not to obtain an optimal partition of the feature space, but rather to provide a sufficiently diverse distribution of candidates from which representative and minimally redundant instances can be selected. Therefore, efficiency and scalability are prioritized over potential improvements in clustering quality.  
Since the *BisectingKMeans* implementation in *MLlib* is distributed, both model fitting and instance assignment to clusters are performed in a distributed manner. As a result, the `clustered_candidates_df` DataFrame is obtained, incorporating the *cluster_id* column indicating the cluster assigned to each instance.  
**Selecting the Instance Closest to Each Cluster Centroid**
--------------------------------------------------------------------

We can observe that the most similar documents are related to the **Bisecting K-Means algorithm**. This demonstrates how semantic retrieval can identify document chunks that are relevant to the user's question, providing the contextual information that an LLM can subsequently use to formulate an answer.

The following notebook explores different retrieval strategies implemented in the `ChromaStore` class, designed to handle different types of user queries and retrieval requirements.